In [11]:

import pandas as pd
import matplotlib.pyplot as plt


In [12]:
df = pd.read_csv("../data/processed/dataset_clean.csv")


In [13]:
df.columns

Index(['cell_id', 'lat', 'lon', 'date', 'year', 'doy', 'label', 'temp_max',
       'temp_min', 'temp_mean', 'rh_min', 'rh_mean', 'wind_max', 'wind_mean',
       'precip_sum', 'vpd_max', 'precip_7d', 'precip_30d',
       'days_since_rain_1mm', 'temp_max_7d_mean', 'rh_min_7d_mean', 'ffmc',
       'dmc', 'dc', 'isi', 'bui', 'fwi', 'days_since_last_fire', 'split',
       'ndvi', 'ndvi_normal', 'ndvi_anomaly', 'ndvi_change_32d',
       'ndvi_stale_days'],
      dtype='str')

In [14]:
from sklearn.model_selection import train_test_split

In [15]:
# Check the predefined chronological split
print(df['split'].value_counts())

# Inspect all columns
print(df.columns.tolist())

split
train    104979
test       8467
val        8423
Name: count, dtype: int64
['cell_id', 'lat', 'lon', 'date', 'year', 'doy', 'label', 'temp_max', 'temp_min', 'temp_mean', 'rh_min', 'rh_mean', 'wind_max', 'wind_mean', 'precip_sum', 'vpd_max', 'precip_7d', 'precip_30d', 'days_since_rain_1mm', 'temp_max_7d_mean', 'rh_min_7d_mean', 'ffmc', 'dmc', 'dc', 'isi', 'bui', 'fwi', 'days_since_last_fire', 'split', 'ndvi', 'ndvi_normal', 'ndvi_anomaly', 'ndvi_change_32d', 'ndvi_stale_days']


In [16]:
features = [
    'doy',
    'temp_max',
    'temp_min',
    'temp_mean',
    'rh_min',
    'rh_mean',
    'wind_max',
    'wind_mean',
    'precip_sum',
    'vpd_max',
    'precip_7d',
    'precip_30d',
    'days_since_rain_1mm',
    'temp_max_7d_mean',
    'rh_min_7d_mean',
    'ffmc',
    'dmc',
    'dc',
    'isi',
    'bui',
    'fwi',
    'ndvi',
    'ndvi_normal',
    'ndvi_change_32d',
    'ndvi_stale_days'
]

print("Feature count:", len(features))

Feature count: 25


In [17]:
train_df = df[df['split'] == 'train'].copy()
val_df = df[df['split'] == 'val'].copy()
test_df = df[df['split'] == 'test'].copy()

x_train = train_df[features]
y_train = train_df['label']

x_val = val_df[features]
y_val = val_df['label']

x_test = test_df[features]
y_test = test_df['label']

print("Train:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)

Train: (104979, 25) (104979,)
Validation: (8423, 25) (8423,)
Test: (8467, 25) (8467,)


In [18]:
missing = x_train.isnull().sum()

print(missing[missing > 0].sort_values(ascending=False))

ndvi_normal        61
ndvi               55
ndvi_change_32d    55
ndvi_stale_days    55
dtype: int64


In [19]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')

x_train_imputed = imputer.fit_transform(x_train)
x_val_imputed = imputer.transform(x_val)
x_test_imputed = imputer.transform(x_test)

In [20]:
for feature, median in zip(features, imputer.statistics_):
    if pd.isna(median):
        print(feature, "-> median is NaN")

In [21]:
for feature, median in zip(features, imputer.statistics_):
    if pd.isna(median):
        print(feature, "-> median is NaN")

In [22]:
from sklearn.preprocessing import StandardScaler
import numpy as np

scaler2 = StandardScaler()

x_train_scaled = scaler2.fit_transform(x_train_imputed)
x_val_scaled = scaler2.transform(x_val_imputed)
x_test_scaled = scaler2.transform(x_test_imputed)

print("Train NaNs:", np.isnan(x_train_scaled).sum())
print("Validation NaNs:", np.isnan(x_val_scaled).sum())

Train NaNs: 0
Validation NaNs: 0


In [23]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import average_precision_score

knn = KNeighborsClassifier(
    n_neighbors=5,
    metric='euclidean',
    weights='uniform',
    n_jobs=-1
)

knn.fit(x_train_scaled, y_train)

y_val_proba = knn.predict_proba(x_val_scaled)[:, 1]

pr_auc = average_precision_score(y_val, y_val_proba)

print("Validation PR-AUC:", round(pr_auc, 4))

Validation PR-AUC: 0.0921


In [24]:
print("Positive rate:", round(y_val.mean(), 4))
print("Negative rate:", round(1 - y_val.mean(), 4))

Positive rate: 0.0763
Negative rate: 0.9237


In [25]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import average_precision_score

for k in [3, 5, 7, 11, 15, 21, 31]:
    knn = KNeighborsClassifier(
        n_neighbors=k,
        metric='euclidean',
        weights='uniform',
        n_jobs=-1
    )

    knn.fit(x_train_scaled, y_train)

    y_val_proba = knn.predict_proba(x_val_scaled)[:, 1]
    score = average_precision_score(y_val, y_val_proba)

    print(f"k={k:2d} | PR-AUC={score:.4f}")

k= 3 | PR-AUC=0.0885
k= 5 | PR-AUC=0.0921
k= 7 | PR-AUC=0.0942
k=11 | PR-AUC=0.0965
k=15 | PR-AUC=0.0969
k=21 | PR-AUC=0.0977
k=31 | PR-AUC=0.0979


In [26]:
for k in [41, 51, 71, 101, 151, 201]:
    knn = KNeighborsClassifier(
        n_neighbors=k,
        metric='euclidean',
        weights='uniform',
        n_jobs=-1
    )

    knn.fit(x_train_scaled, y_train)

    y_val_proba = knn.predict_proba(x_val_scaled)[:, 1]
    score = average_precision_score(y_val, y_val_proba)

    print(f"k={k:3d} | PR-AUC={score:.4f}")

k= 41 | PR-AUC=0.0994
k= 51 | PR-AUC=0.0998
k= 71 | PR-AUC=0.1009
k=101 | PR-AUC=0.1012
k=151 | PR-AUC=0.1016
k=201 | PR-AUC=0.1023


In [27]:
for k in [301, 401, 501, 701, 1001]:
    knn = KNeighborsClassifier(
        n_neighbors=k,
        metric='euclidean',
        weights='uniform',
        n_jobs=-1
    )

    knn.fit(x_train_scaled, y_train)

    y_val_proba = knn.predict_proba(x_val_scaled)[:, 1]
    score = average_precision_score(y_val, y_val_proba)

    print(f"k={k:4d} | PR-AUC={score:.4f}")

k= 301 | PR-AUC=0.1026
k= 401 | PR-AUC=0.1025
k= 501 | PR-AUC=0.1027
k= 701 | PR-AUC=0.1033
k=1001 | PR-AUC=0.1043


In [28]:
for k in [1501, 2001, 3001, 4001, 5001]:
    knn = KNeighborsClassifier(
        n_neighbors=k,
        metric='euclidean',
        weights='uniform',
        n_jobs=-1
    )

    knn.fit(x_train_scaled, y_train)

    y_val_proba = knn.predict_proba(x_val_scaled)[:, 1]
    score = average_precision_score(y_val, y_val_proba)

    print(f"k={k:4d} | PR-AUC={score:.4f}")

k=1501 | PR-AUC=0.1041
k=2001 | PR-AUC=0.1046
k=3001 | PR-AUC=0.1042
k=4001 | PR-AUC=0.1042
k=5001 | PR-AUC=0.1045


In [29]:
for weights in ['uniform', 'distance']:
    knn = KNeighborsClassifier(
        n_neighbors=2001,
        metric='euclidean',
        weights=weights,
        n_jobs=-1
    )

    knn.fit(x_train_scaled, y_train)

    y_val_proba = knn.predict_proba(x_val_scaled)[:, 1]
    score = average_precision_score(y_val, y_val_proba)

    print(f"weights={weights:8s} | PR-AUC={score:.4f}")

weights=uniform  | PR-AUC=0.1046
weights=distance | PR-AUC=0.1044


In [30]:
knn = KNeighborsClassifier(
    n_neighbors=2001,
    metric='manhattan',
    weights='uniform',
    n_jobs=-1
)

knn.fit(x_train_scaled, y_train)

y_val_proba = knn.predict_proba(x_val_scaled)[:, 1]
score = average_precision_score(y_val, y_val_proba)

print("Manhattan PR-AUC:", round(score, 4))

Manhattan PR-AUC: 0.1027
